In [ ]:
# make local editable packages automatically reload
%load_ext autoreload
%autoreload 2

# Qualitative Evaluation omnipose

In [ ]:
from omnipose.gpu import use_gpu
use_GPU = use_gpu()
use_GPU

In [ ]:
import omnipose

# set up plotting defaults
omnipose.plot.setup()

# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams['figure.dpi'] = 300
plt.style.use('dark_background')
%matplotlib inline

## Testing channel combination

### Model configuration

In [ ]:
import os
from cellpose_omni import models

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# model_name = 'bact_phase_affinity'
# model_name = 'worm_high_res_omni'
# model_name = 'worm_bact_omni'
# model_name = 'cyto2_omni'
# model_name = 'bact_phase_cp'
# model_name = 'bact_fluor_cp'
# model_name = 'cyto2'
model_name = 'cyto'
model = models.CellposeModel(gpu=use_GPU, model_type=model_name)

### File pre-loading

In [ ]:
# Import the necessary classes from your project files
from image_processing_tools.image_class.image_container import ImageContainer
from image_processing_tools.util.load_files import find_files_by_pattern

# 1. Define the configuration dictionary.
# This should contain preprocessing settings needed by ImageContainer.
config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1024,
        "outlier_percentile": 0.35,
        "quantization": "16bit",
        "correct_DIC_shift": [0,0],
    }
}

# 2. Define the search directory and file pattern.
search_path = (
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV1_1_CY5, CY3.5 NAR, FITC, DAPI, BF",
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV2_1_CY5, CY3.5 NAR, FITC, DAPI",
    "~/A8/Data_Roan/250925_Cocu_Dyes_cFOS_CSF3_ECE1/Cocu_Cellbrite_cFOS_CSF3_ECE1_01_CY5, CY3.5 NAR, CY3, FITC, DAPI"
    )

file_pattern = (
    "*MAX*.tif",
    "*SHARPEST-normalized_variance*.tif"
    )

# 3. Find the files. The 'files' variable will hold the list of Path objects.
files = find_files_by_pattern(search_path[2], file_pattern[1], verbose=True)

# 4. Create a list of mono-channel ImageContainer objects from the file paths.
channel_images = [ImageContainer(path, config) for path in files]

# 5. Create composite images
# TODO: For new datasets change DAPI_INDEX
DAPI_INDEX = 4
dapi_channel_obj = channel_images[DAPI_INDEX]

composite_images = []
for i, other_channel_obj in enumerate(channel_images):
    if i == DAPI_INDEX:
        continue # Skip the DAPI channel itself

    # Use the overloaded '+' operator to create a composite ImageContainer
    # containing the DAPI channel and the current 'other_channel_obj'.
    composite = other_channel_obj + dapi_channel_obj 
    composite_images.append(composite)
    print(f"Created composite image: {dapi_channel_obj.channels[0].path.name} + {other_channel_obj.channels[0].path.name}")

print(f"\nSuccessfully created {len(composite_images)} composite images.")


In [ ]:
from typing import Any, Dict, List, Optional, Union
import numpy as np
import cv2
import re
from pathlib import Path

def prepare_analysis_image(image_input: 'ImageContainer') -> np.ndarray:
        """
        Creates the final merged image for model input.

        Args:
            image_input: An ImageContainer (mono-channel or composite).

        Returns:
            The merged image.
        """
        return image_input.merge()

def get_composite_channel_names(
    composite_images: List['ImageContainer'], 
    as_string: bool = False, 
    separator: str = '_'
) -> Union[List[List[str]], List[str]]:
    """
    Extracts channel names (e.g., 'C1', 'DAPI') from a list of composite ImageContainer objects.

    This function inspects each ImageContainer to find the file paths of its
    constituent channels and extracts a channel identifier (like 'C1', 'C2', etc.)
    from each filename using a regular expression.

    Args:
        composite_images (List[ImageContainer]): A list of composite ImageContainer objects to process.
        as_string (bool, optional): If True, joins the channel names for each composite
                                    into a single string. Defaults to False.
        separator (str, optional): The separator to use when joining names if as_string is True.
                                   Defaults to '_'.

    Returns:
        Union[List[List[str]], List[str]]: 
        - If as_string is False (default), returns a list of lists, where each inner
          list contains the channel names for one composite image (e.g., [['C1', 'DAPI'], ['C2', 'DAPI']]).
        - If as_string is True, returns a list of strings (e.g., ['C1_DAPI', 'C2_DAPI']).
    """
    
    def _get_channel_name_from_path(file_path: Path) -> str:
        """Helper to extract a channel identifier like 'C1' from a filename."""
        # Matches the first occurrence of 'C' followed by one or more digits,
        # or 'DAPI' case-insensitively.
        match = re.search(r'(C\d+)', file_path.name, re.IGNORECASE)
        if match:
            # Return the found group, converted to uppercase for consistency (e.g., 'dapi' -> 'DAPI')
            return match.group(1).upper()
        # Fallback if no specific channel pattern is found
        return file_path.stem

    all_channel_names = []
    for composite in composite_images:
        # Get the list of channel objects from the composite ImageContainer
        channels = composite.channels
        # Extract the name from the path of each channel
        names = [_get_channel_name_from_path(ch.path) for ch in channels]
        
        if as_string:
            all_channel_names.append(separator.join(names))
        else:
            all_channel_names.append(names)
            
    return all_channel_names

In [ ]:
import time

# define parameters
params = {'channels':None, # always define this if using older models, e.g. [0,0] with bact_phase_omni
          'rescale': None, # upscale or downscale your images, None = no rescaling 
          'mask_threshold': -2, # erode or dilate masks with higher or lower values between -5 and 5 
          'flow_threshold': 0, # default is .4, but only needed if there are spurious masks to clean up; slows down output
          'transparency': True, # transparency in flow output
          'omni': False, # we can turn off Omnipose mask reconstruction, not advised 
          'cluster': True, # use DBSCAN clustering
          'resample': True, # whether or not to run dynamics on rescaled grid or original grid 
          'verbose': False, # turn on if you want to see more output 
          'tile': True, # average the outputs from flipped (augmented) images; slower, usually not needed 
          'niter': None, # default None lets Omnipose calculate # of Euler iterations (usually <20) but you can tune it for over/under segmentation 
          'augment': False, # Can optionally rotate the image and average network outputs, usually not needed 
          'affinity_seg': False, # new feature, stay tuned...
          'mask_threshold': -2,
          'diameter': 80
         }

imgs = [prepare_analysis_image(i) for i in composite_images]

tic = time.time() 
masks, flows, styles = model.eval(imgs, **params)

net_time = time.time() - tic

print('total segmentation time: {}s'.format(net_time))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- Setup output paths ---
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/omnipose_composite_visualizations")
relative_dir = files[0].parent.relative_to(base_input_dir)
current_output_dir = base_output_dir / relative_dir
current_output_dir.mkdir(parents=True, exist_ok=True)

channel_names = get_composite_channel_names(composite_images, as_string=True, separator='+')

def _composite_to_rgb(img: np.ndarray) -> np.ndarray:
    """Best-effort RGB conversion for display of 1/2/3-channel composite images."""
    if img.ndim == 2:
        return img
    n = img.shape[-1]
    if n == 2:
        ch0 = img[..., 0].astype(np.float32)
        ch1 = img[..., 1].astype(np.float32)
        ch0 = (ch0 - ch0.min()) / (np.ptp(ch0) + 1e-8)
        ch1 = (ch1 - ch1.min()) / (np.ptp(ch1) + 1e-8)
        return np.stack([ch0, ch1, (ch0 + ch1) / 2], axis=-1)
    return img[..., :3]

# The threshold actually used by mask reconstruction for this run (dynamics.compute_masks
# does `cp_mask = cellprob > mask_threshold` when omni=False; omnipose.core.compute_masks
# thresholds the distance field the same way when omni=True).
mask_threshold = params['mask_threshold']

# --- Build the comparison figure: one row per image combination, 4 columns ---
n_images = len(imgs)
fig, axes = plt.subplots(n_images, 4, figsize=(20, 4 * n_images), constrained_layout=True)
if n_images == 1:
    axes = axes.reshape(1, -1)

for idx, img in enumerate(imgs):
    maski = masks[idx]
    flowi = flows[idx][0]      # RGB flow field
    cellprobi = flows[idx][2]  # cell probability / distance field
    channel_comp = channel_names[idx]

    # 1. Image + mask overlay
    ax_img = axes[idx, 0]
    ax_img.imshow(_composite_to_rgb(img))
    if maski.max() > 0:
        ax_img.imshow(maski, cmap='nipy_spectral', alpha=0.4, interpolation='nearest')
    ax_img.set_title(f"{channel_comp}\nImage + Mask")
    ax_img.axis('off')

    # 2. Flow field
    ax_flow = axes[idx, 1]
    ax_flow.imshow(flowi)
    ax_flow.set_title("Flow Field")
    ax_flow.axis('off')

    # 3. Cell probability
    ax_prob = axes[idx, 2]
    ax_prob.imshow(cellprobi, cmap='gray')
    ax_prob.set_title("Cell Probability")
    ax_prob.axis('off')

    # 4. Thresholded cell probability (same threshold used in mask reconstruction)
    ax_thresh = axes[idx, 3]
    ax_thresh.imshow(cellprobi > mask_threshold, cmap='gray')
    ax_thresh.set_title(f"Thresholded Cell Probability\n(mask_threshold={mask_threshold})")
    ax_thresh.axis('off')

fig.suptitle(f"Omnipose Segmentation Results ({model_name})", fontsize=16)

output_filename = current_output_dir / f"{model_name}_omnipose_composite_comparison.png"
plt.savefig(output_filename, dpi=300)
plt.close(fig)
print(f"Saved comparison figure to: {output_filename}")

## Test all models, make composite figure

In [ ]:
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from cellpose_omni import models
from cellpose_omni.models import CP_MODELS, C2_MODELS, C2_BD_MODELS, C1_BD_MODELS, C1_MODELS, C2_MODEL_NAMES

# Cellpose-trained vs Omnipose-trained model registries, taken directly from
# cellpose_omni/models.py + omnipose/core.py. Using the library's own model
# lists (rather than a naive `'omni' in model_name` check) matters because
# 'bact_phase_affinity' is an Omnipose-family model with no "omni" in its name.
CELLPOSE_MODEL_NAMES = set(CP_MODELS + C2_MODELS)
OMNIPOSE_MODEL_NAMES = set(C2_BD_MODELS + C1_BD_MODELS + C1_MODELS)

def is_omni_model(name: str) -> bool:
    if name in OMNIPOSE_MODEL_NAMES:
        return True
    if name in CELLPOSE_MODEL_NAMES:
        return False
    raise ValueError(f"Unknown model '{name}'; add it to CELLPOSE_MODEL_NAMES or OMNIPOSE_MODEL_NAMES.")

def is_mono_channel_model(name: str) -> bool:
    """True if the model expects nchan=1 (see CellposeModel.__init__: nchan only
    becomes 2 when model_type is in C2_MODEL_NAMES)."""
    return name not in C2_MODEL_NAMES

model_names = [
    'bact_phase_affinity',
    'bact_phase_omni',
    'bact_fluor_omni',
    'worm_omni',
    'worm_high_res_omni',
    'worm_bact_omni',
    'cyto2_omni',
    'bact_phase_cp',
    'bact_fluor_cp',
    'cyto2',
    'cyto',
    # plant_omni intentionally excluded: it's a native 3D model (dim=3, z-stack
    # input) requiring ~20GB VRAM, per 7_omnipose_3d.ipynb -- not comparable to
    # the single-2D-image eval done here.
]

base_params = {
    'channels': None,
    'rescale': None,
    'mask_threshold': -2,
    'flow_threshold': 0,
    'transparency': True,
    'cluster': True,        # only takes effect when omni=True
    'resample': True,
    'verbose': False,
    'tile': True,
    'niter': None,
    'augment': False,
    'affinity_seg': False,  # only takes effect when omni=True
    'diameter': 80,
}

def _sum_channels_rescaled(img: np.ndarray) -> np.ndarray:
    """Sum a multi-channel image along the last axis and rescale back to the
    input dtype's range (same min-max stretch ImageContainer._sum_channels uses)."""
    if img.ndim == 2:
        return img
    dtype = img.dtype
    summed = img.astype(np.float64).sum(axis=-1)
    min_val, max_val = summed.min(), summed.max()
    if max_val > min_val:
        d_info = np.iinfo(dtype)
        summed = ((summed - min_val) / (max_val - min_val) * d_info.max).astype(dtype)
    else:
        summed = np.zeros_like(summed, dtype=dtype)
    return summed

# --- Pick one image to run every model on ---
test_comp_img = composite_images[1]
test_channel_name = get_composite_channel_names([test_comp_img], as_string=True)[0]
test_img = prepare_analysis_image(test_comp_img)          # (H, W, 2) for 2-channel models
test_img_mono = _sum_channels_rescaled(test_img)           # (H, W) summed+rescaled, for 1-channel models

# --- Setup output paths ---
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/omnipose_composite_visualizations")
relative_dir = files[0].parent.relative_to(base_input_dir)
current_output_dir = base_output_dir / relative_dir
current_output_dir.mkdir(parents=True, exist_ok=True)

n_models = len(model_names)
fig, axes = plt.subplots(n_models, 4, figsize=(20, 4 * n_models), constrained_layout=True)
if n_models == 1:
    axes = axes.reshape(1, -1)

print(f"Running {n_models} models on '{test_channel_name}'...")

for i, name in enumerate(model_names):
    omni = is_omni_model(name)
    mono = is_mono_channel_model(name)
    model_input = test_img_mono if mono else test_img
    print(f"[{i+1}/{n_models}] {name} (omni={omni}, mono={mono})")

    model_i = models.CellposeModel(gpu=use_GPU, model_type=name)

    params_i = dict(base_params, omni=omni)
    tic = time.time()
    maski, flowsi, _ = model_i.eval(model_input, **params_i)
    print(f"  -> {time.time() - tic:.2f}s")

    flowi = flowsi[0]       # RGB flow field
    cellprobi = flowsi[2]   # cell probability / distance field
    mask_threshold = params_i['mask_threshold']

    ax_img, ax_flow, ax_prob, ax_thresh = axes[i]

    ax_img.imshow(_composite_to_rgb(model_input))
    if maski.max() > 0:
        ax_img.imshow(maski, cmap='nipy_spectral', alpha=0.4, interpolation='nearest')
    ax_img.set_title(f"{name} (omni={omni}, mono={mono})\nImage + Mask")
    ax_img.axis('off')

    ax_flow.imshow(flowi)
    ax_flow.set_title("Flow Field")
    ax_flow.axis('off')

    ax_prob.imshow(cellprobi, cmap='gray')
    ax_prob.set_title("Cell Probability")
    ax_prob.axis('off')

    ax_thresh.imshow(cellprobi > mask_threshold, cmap='gray')
    ax_thresh.set_title(f"Thresholded Cell Probability\n(mask_threshold={mask_threshold})")
    ax_thresh.axis('off')

fig.suptitle(f"Omnipose/Cellpose Model Comparison — {test_channel_name}", fontsize=16)

output_filename = current_output_dir / f"all_models_{test_channel_name}_comparison.png"
plt.savefig(output_filename, dpi=300)
plt.close(fig)
print(f"Saved comparison figure to: {output_filename}")

## Testing different parameters

### Diameter

In [ ]:
test_comp_img = composite_images[1]
diameters = [10,30,50,80,120,150,180]
# diameters = [200,220,240,260,280,300,320]

# define parameters
params = {'channels':None, # always define this if using older models, e.g. [0,0] with bact_phase_omni
          'rescale': None, # upscale or downscale your images, None = no rescaling 
          'mask_threshold': -2, # erode or dilate masks with higher or lower values between -5 and 5 
          'flow_threshold': 0, # default is .4, but only needed if there are spurious masks to clean up; slows down output
          'transparency': True, # transparency in flow output
          'omni': is_omni_model(model_name), # set automatically from the model registry (see "Test all models" section)
          'cluster': True, # use DBSCAN clustering
          'resample': True, # whether or not to run dynamics on rescaled grid or original grid 
          'verbose': False, # turn on if you want to see more output 
          'tile': True, # average the outputs from flipped (augmented) images; slower, usually not needed 
          'niter': None, # default None lets Omnipose calculate # of Euler iterations (usually <20) but you can tune it for over/under segmentation 
          'augment': False, # Can optionally rotate the image and average network outputs, usually not needed 
          'affinity_seg': False, # new feature, stay tuned...
          'mask_threshold': -2,
          'diameter': 80
         }

test_merged_img = prepare_analysis_image(test_comp_img)

diam_test_masks = {}

for diam in diameters:
    params['diameter'] = diam
    masks, flows, styles = model.eval(test_merged_img, **params)
    diam_test_masks[diam] = masks


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.color import label2rgb

def normalize01(x):
    x = x.astype(float)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

def two_channel_to_rgb(img2):
    """
    Convert (H,W,2) → RGB:
    R = channel 0
    G = channel 1
    B = average
    """
    ch0 = normalize01(img2[..., 0])
    ch1 = normalize01(img2[..., 1])
    ch_mean = (ch0 + ch1) / 2
    return np.stack([ch0, ch1, ch_mean], axis=-1)

# Convert base image
base_rgb = two_channel_to_rgb(test_merged_img)

# Prepare figure
fig, axes = plt.subplots(1, len(diameters), figsize=(5 * len(diameters), 5))
if len(diameters) == 1:
    axes = [axes]

for ax, diam in zip(axes, diameters):
    mask = diam_test_masks[diam]

    # ----- Mask fill overlay (this colors FULL mask regions) -----
    # label2rgb assigns one color per label and blends it with background
    mask_rgb = label2rgb(mask, image=base_rgb, bg_label=0, alpha=0.4)

    ax.imshow(mask_rgb)
    ax.set_title(f"diam={diam}", fontsize=12)
    ax.axis("off")

plt.tight_layout()

# --- Save to disk (same naming convention as the other comparison figures) ---
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/omnipose_composite_visualizations")
relative_dir = files[0].parent.relative_to(base_input_dir)
current_output_dir = base_output_dir / relative_dir
current_output_dir.mkdir(parents=True, exist_ok=True)

test_channel_name = get_composite_channel_names([test_comp_img], as_string=True)[0]
output_filename = current_output_dir / f"{model_name}_diameter_sweep_{test_channel_name}.png"
plt.savefig(output_filename, dpi=300)
print(f"Saved diameter sweep figure to: {output_filename}")

# plt.show()


### Mask threshold

In [ ]:
test_comp_img = composite_images[1]
mask_thresholds = [-5,-4,-3,-2,-1,0,2,3]

# define parameters
params = {'channels':None, # always define this if using older models, e.g. [0,0] with bact_phase_omni
          'rescale': None, # upscale or downscale your images, None = no rescaling 
          'mask_threshold': -2, # erode or dilate masks with higher or lower values between -5 and 5 
          'flow_threshold': 0, # default is .4, but only needed if there are spurious masks to clean up; slows down output
          'transparency': True, # transparency in flow output
          'omni': is_omni_model(model_name), # set automatically from the model registry (see "Test all models" section)
          'cluster': True, # use DBSCAN clustering
          'resample': True, # whether or not to run dynamics on rescaled grid or original grid 
          'verbose': False, # turn on if you want to see more output 
          'tile': True, # average the outputs from flipped (augmented) images; slower, usually not needed 
          'niter': None, # default None lets Omnipose calculate # of Euler iterations (usually <20) but you can tune it for over/under segmentation 
          'augment': False, # Can optionally rotate the image and average network outputs, usually not needed 
          'affinity_seg': False, # new feature, stay tuned...
          'mask_threshold': -2,
          'diameter': 150
         }

test_merged_img = prepare_analysis_image(test_comp_img)

mask_test_masks = {}

for mask_thresh in mask_thresholds:
    params['mask_threshold'] = mask_thresh
    masks, flows, styles = model.eval(test_merged_img, **params)
    mask_test_masks[mask_thresh] = masks

In [ ]:
from pathlib import Path

# Recompute from this section's own test image (composite_images[3]), not the
# diameter sweep's `base_rgb` (composite_images[1]) which this cell used to
# silently reuse via the shared global.
base_rgb = two_channel_to_rgb(test_merged_img)

# Prepare figure
fig, axes = plt.subplots(1, len(mask_thresholds), figsize=(5 * len(mask_thresholds), 5))
if len(mask_thresholds) == 1:
    axes = [axes]

for ax, diam in zip(axes, mask_thresholds):
    mask = mask_test_masks[diam]

    # ----- Mask fill overlay (this colors FULL mask regions) -----
    # label2rgb assigns one color per label and blends it with background
    mask_rgb = label2rgb(mask, image=base_rgb, bg_label=0, alpha=0.4)

    ax.imshow(mask_rgb)
    ax.set_title(f"Mask Threshold={diam}", fontsize=12)
    ax.axis("off")

plt.tight_layout()

# --- Save to disk (same naming convention as the other comparison figures) ---
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/omnipose_composite_visualizations")
relative_dir = files[0].parent.relative_to(base_input_dir)
current_output_dir = base_output_dir / relative_dir
current_output_dir.mkdir(parents=True, exist_ok=True)

test_channel_name = get_composite_channel_names([test_comp_img], as_string=True)[0]
output_filename = current_output_dir / f"{model_name}_mask_threshold_sweep_{test_channel_name}.png"
plt.savefig(output_filename, dpi=300)
print(f"Saved mask threshold sweep figure to: {output_filename}")

# plt.show()

## Test channel ordering

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- Select composite image (uses the currently loaded `model`/`model_name`) ---
test_comp_img = composite_images[1]  # change index to test a different composite
test_channel_name = get_composite_channel_names([test_comp_img], as_string=True)[0]

params = {'channels': None,
          'rescale': None,
          'mask_threshold': -2,
          'flow_threshold': 0,
          'transparency': True,
          'omni': is_omni_model(model_name), # set automatically from the model registry
          'cluster': True,
          'resample': True,
          'verbose': False,
          'tile': True,
          'niter': None,
          'augment': False,
          'affinity_seg': False,
          'diameter': 80,
         }

# --- Build default-order and swapped-order composites ---
img_default = prepare_analysis_image(test_comp_img)                # (H, W, 2), as composed: [other_channel, DAPI]
img_swapped = np.ascontiguousarray(img_default[..., ::-1])          # reverse channel order: [DAPI, other_channel]

orderings = {
    'default order': img_default,
    'swapped order': img_swapped,
}

# --- Run the selected model on both orderings ---
order_masks = {}
for label, img in orderings.items():
    maski, flowsi, _ = model.eval(img, **params)
    order_masks[label] = maski

# --- Comparison figure: mask overlay only ---
fig, axes = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
for ax, (label, img) in zip(axes, orderings.items()):
    maski = order_masks[label]
    ax.imshow(_composite_to_rgb(img))
    if maski.max() > 0:
        ax.imshow(maski, cmap='nipy_spectral', alpha=0.4, interpolation='nearest')
    ax.set_title(f"{label}\n({maski.max()} cells)")
    ax.axis('off')

fig.suptitle(f"Channel Ordering Comparison ({model_name}, {test_channel_name})", fontsize=14)

# --- Save to disk (same naming convention as the other comparison figures) ---
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/omnipose_composite_visualizations")
relative_dir = files[0].parent.relative_to(base_input_dir)
current_output_dir = base_output_dir / relative_dir
current_output_dir.mkdir(parents=True, exist_ok=True)

output_filename = current_output_dir / f"{model_name}_channel_order_comparison_{test_channel_name}.png"
plt.savefig(output_filename, dpi=300)
print(f"Saved channel ordering comparison figure to: {output_filename}")
# plt.show()